# 05 — Broader Economic Context
## Trade Winds & Headwinds: USVI, 2002–2022

Sections:
1. USVI nominal and real GDP
2. Accommodation & food services: nominal value added
3. Real value added (BEA chained dollars)
4. Industry GDP share
5. GDP composition note

**Coverage:** BEA data ends 2021 for industry breakdowns and 2022 for GDP totals.  
**No CPI deflation.** Real-dollar series use BEA's own chained-dollar measures only.  
**Important:** Accommodation & food services ≠ tourism GDP. It excludes tourism-related retail, transportation, and recreation, and includes resident consumption.

In [1]:
import sys, importlib
from pathlib import Path
import os

PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import src.cleaning as _c
import src.charts as _ch

importlib.reload(_c)
importlib.reload(_ch)

from src.cleaning import load_bea, load_usviber_combined
from src.charts import PALETTE, line_chart, bar_chart

pd.set_option("display.float_format", "{:,.1f}".format)
print("Ready.")

Ready.


In [2]:
bea_nom = load_bea("2.1")  # nominal value added
bea_real = load_bea("2.3")  # real (chained) value added
bea_gdp = load_bea("1.1")  # nominal GDP expenditure accounts
bea_rgdp = load_bea("1.2")  # real GDP
tour = load_usviber_combined()


def _get(df, pattern):
    """Extract a unique label row from a BEA table."""
    rows = df[df["label"].str.contains(pattern, case=False, na=False)]
    # Keep the first match per year (some tables have duplicates from stacked sheets)
    return rows.drop_duplicates(subset="year").sort_values("year")


gdp_nom = _get(bea_gdp, "Gross domestic product")[
    ["year", "value", "vintage_flag"]
].rename(columns={"value": "gdp_nom"})
gdp_real = _get(bea_rgdp, "Gross domestic product")[
    ["year", "value", "vintage_flag"]
].rename(columns={"value": "gdp_real"})
af_nom = _get(bea_nom, "Accommodation")[["year", "value", "vintage_flag"]].rename(
    columns={"value": "af_nom"}
)
af_real = _get(bea_real, "Accommodation")[["year", "value", "vintage_flag"]].rename(
    columns={"value": "af_real"}
)

print("Nominal GDP years:", sorted(gdp_nom.year.tolist()))
print("Real GDP years:   ", sorted(gdp_real.year.tolist()))
print("A&F nom years:    ", sorted(af_nom.year.tolist()))
print("A&F real years:   ", sorted(af_real.year.tolist()))

Nominal GDP years: [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Real GDP years:    [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
A&F nom years:     [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
A&F real years:    [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


---
## 1. USVI GDP

In [3]:
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=gdp_nom["year"],
        y=gdp_nom["gdp_nom"],
        name="Nominal GDP ($M)",
        marker_color=PALETTE["gdp"],
    )
)
fig.update_layout(
    title=dict(
        text="USVI Nominal GDP ($M)<br>"
        '<span style="font-size:13px;color:#555">BEA Table 1.1. Gross Domestic Product, millions of current dollars.</span>',
        font=dict(size=17),
    ),
    yaxis_title="GDP ($M, nominal)",
    yaxis_tickformat=",",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: BEA USVI GDP data (vigdp0624.xlsx)</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

In [4]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=gdp_real["year"],
        y=gdp_real["gdp_real"],
        mode="lines+markers",
        name="Real GDP (chained $M)",
        line=dict(color=PALETTE["gdp"], width=2.5),
    )
)
fig.update_layout(
    title=dict(
        text="USVI Real GDP (Chained Dollars)<br>"
        '<span style="font-size:13px;color:#555">BEA Table 1.2. Millions of chained (2012) dollars. BEA-constructed deflator.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Real GDP ($M, chained)",
    yaxis_tickformat=",",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: BEA USVI GDP data (vigdp0624.xlsx)</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()
print("Real GDP note: Values in chained (2012) dollars. Federal transfers and CDBG-DR")
print(
    "construction spending kept nominal and real GDP elevated post-2017 even as tourism collapsed."
)

Real GDP note: Values in chained (2012) dollars. Federal transfers and CDBG-DR
construction spending kept nominal and real GDP elevated post-2017 even as tourism collapsed.


---
## 2. Accommodation & Food Services — Nominal Value Added

In [5]:
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=af_nom["year"],
        y=af_nom["af_nom"],
        name="A&F value added ($M)",
        marker_color=PALETTE["value_added"],
        # Mark revised values with lower opacity
        marker_opacity=[0.6 if r else 1.0 for r in af_nom["vintage_flag"].eq("r")],
    )
)
fig.update_layout(
    title=dict(
        text="Accommodation & Food Services — Nominal Value Added ($M)<br>"
        '<span style="font-size:13px;color:#555">BEA Table 2.1. Faded bars = revised estimates.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Value added ($M, nominal)",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: BEA USVI GDP data (vigdp0624.xlsx). A&F ≠ tourism GDP.</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

---
## 3. Real Value Added

In [6]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=af_real["year"],
        y=af_real["af_real"],
        mode="lines+markers",
        name="A&F real value added (chained $M)",
        line=dict(color=PALETTE["value_added"], width=2.5),
        marker=dict(
            size=6,
            color=[
                PALETTE["cruise"] if r else PALETTE["value_added"]
                for r in af_real["vintage_flag"].eq("r")
            ],
        ),
    )
)
fig.update_layout(
    title=dict(
        text="Accommodation & Food Services — Real Value Added (Chained Dollars)<br>"
        '<span style="font-size:13px;color:#555">BEA Table 2.3. Orange markers = revised. Coverage: 2002–2021.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Real value added ($M, chained)",
    height=440,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: BEA USVI GDP data (vigdp0624.xlsx)</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()
print(af_real.to_string(index=False))

 year  af_real vintage_flag
 2002    342.0             
 2003    360.0             
 2004    380.0             
 2005    387.0             
 2006    382.0             
 2007    450.0             
 2008    470.0             
 2009    416.0             
 2010    445.0             
 2011    454.0             
 2012    450.0             
 2013    441.0             
 2014    444.0             
 2015    438.0             
 2016    428.0             
 2017    376.0             
 2018    240.0            r
 2019    274.0            r
 2020    200.0            r
 2021    373.0             


---
## 4. Industry GDP Share

In [7]:
# Compute share from nominal tables
share = af_nom[["year", "af_nom"]].merge(gdp_nom[["year", "gdp_nom"]], on="year")
share["af_pct"] = (share["af_nom"] / share["gdp_nom"] * 100).round(1)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=share["year"],
        y=share["af_pct"],
        mode="lines+markers",
        name="A&F share of nominal GDP (%)",
        line=dict(color=PALETTE["value_added"], width=2.5),
        fill="tozeroy",
        fillcolor="rgba(26,79,114,0.1)",
    )
)
fig.update_layout(
    title=dict(
        text="Accommodation & Food Services as Share of USVI GDP<br>"
        '<span style="font-size:13px;color:#555">Nominal value added / nominal GDP. Coverage ends 2021.</span>',
        font=dict(size=17),
    ),
    yaxis_title="A&F share of GDP (%)",
    yaxis_range=[0, 20],
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: BEA USVI GDP data — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()
print(share.set_index("year").to_string())

      af_nom  gdp_nom  af_pct
year                         
2002   267.0  3,262.0     8.2
2003   284.0  3,443.0     8.2
2004   308.0  3,797.0     8.1
2005   325.0  4,428.0     7.3
2006   330.0  4,484.0     7.4
2007   405.0  4,784.0     8.5
2008   424.0  4,244.0    10.0
2009   400.0  4,201.0     9.5
2010   428.0  4,324.0     9.9
2011   436.0  4,223.0    10.3
2012   450.0  4,089.0    11.0
2013   454.0  3,738.0    12.1
2014   470.0  3,565.0    13.2
2015   486.0  3,663.0    13.3
2016   483.0  3,798.0    12.7
2017   436.0  3,794.0    11.5
2018   288.0  3,923.0     7.3
2019   341.0  4,121.0     8.3
2020   253.0  4,229.0     6.0
2021   495.0  4,507.0    11.0


---
## 5. GDP Composition Note

**Why did USVI GDP grow after the 2017 hurricanes and during COVID-19?**

USVI received approximately \$8.3 billion in CDBG-DR and other federal disaster recovery funds after Irma and Maria — larger than twice the territory's annual GDP. This spending flowed through construction, government transfers, and private reconstruction activity, inflating both nominal and real GDP measures even as the tourism sector collapsed.

This means that broad GDP is a misleading indicator of tourism-sector health. The accommodation & food services value-added share — which fell from ~13% in 2015–2016 to ~6% in 2020 — more accurately captures the sector's contraction relative to an economy distorted by federal relief flows.

**Implication for analysis:** Do not use total GDP as a tourism recovery indicator. Use the sector-specific series (A&F value added, FRED L&H employment, USVIBER visitor statistics) instead.